In [2]:
import pandas as pd
import os
import itertools
import numpy as np

In [31]:
df = pd.read_csv('monomer_data/monomers.csv')
df['mon_class'] = df['mon_class'].map(lambda x: str(x).lower())
df = df[df['mon_class'].isin(["cationic", "hydrophobic", "hydrophilic"])]
import json

encoding={v:chr(65+i) for i, v in enumerate(df["mon_abb"])}
json.dump(encoding, open("monomer_data/monomer_encoding.json", 'w'))

In [3]:
df = pd.read_csv('monomer_data/monomers.csv')
df['mon_class'] = df['mon_class'].map(lambda x: str(x).lower())
df = df[df['mon_class'].isin(["cationic", "hydrophobic", "hydrophilic"])]
df

,mon_class,mon_name,mon_abb,SMILES,min_class_wt,max_class_wt,min_mon_wt,max_mon_wt,increments,molar_mass,notes
0,cationic,(3-acrylamidopropyl)trimethylammonium chloride,Tma,C[N+](C)(C)CCCNC(=O)C=C,10.0,90.0,10.0,90.0,5.0,206.71,FOR ALL MONOMERS: the C=C double bond goes awa...
1,cationic,guanidinoethyl acrylamide,Aeg,[Cl-].NC(=[NH2+])NCCNC(=O)C=C,10.0,90.0,0.0,20.0,5.0,192.65,"Haven't used this yet, but I could try it movi..."
2,hydrophilic,4-acryloylmorpholine,Mo,C=CC(=O)N1CCOCC1,0.0,80.0,0.0,90.0,5.0,141.17,NaN
3,hydrophilic,N-(3-methoxypropyl)acrylamide,Mep,C=CC(=O)NCCCOC,0.0,80.0,0.0,90.0,5.0,143.18,NaN
4,hydrophobic,N-Isopropylacrylamide,Ni,CC(C)NC(=O)C=C,10.0,45.0,0.0,40.0,5.0,113.16,NaN
5,hydrophobic,N-phenylacrylamide,Phe,C=CC(=O)Nc1ccccc1,10.0,45.0,0.0,40.0,5.0,147.17,NaN
6,hydrophobic,N-dodecylacrylamide,Do,CCCCCCCCCCCCNC(=O)C=C,10.0,45.0,0.0,30.0,5.0,239.40,NaN
7,hydrophobic,N-butylacrylamide,Bam,CCCCOCNC(=O)C=C,10.0,45.0,0.0,40.0,5.0,127.18,NaN
8,hydrophobic,N-octylacrylamide,Oct,CCCCCCCCNC(=O)C=C,10.0,45.0,0.0,30.0,5.0,183.29,NaN
9,hydrophobic,oleylacrylamide,Olam,CCCCCCCC/C=C\CCCCCCCCNC(=O)C(=C),10.0,45.0,0.0,30.0,5.0,321.30,this is the only monomer that has a C=C bond t...


In [4]:
monomer_classes = {}

for mon_class in df['mon_class'].unique():
    
    mon_class_dict = {}
    idx = (df['mon_class'] == mon_class)
    mon_class_dict['monomers'] = df[idx].mon_abb.tolist()
    mon_class_dict['min'] = df[idx].min_class_wt.unique().item()
    mon_class_dict['max'] = df[idx].max_class_wt.unique().item()
    mon_class_dict['increment'] = df[idx].increments.unique().item()
    monomer_classes[mon_class] = mon_class_dict

monomer_classes

{'cationic': {'monomers': ['Tma', 'Aeg'],
  'min': 10.0,
  'max': 90.0,
  'increment': 5.0},
 'hydrophilic': {'monomers': ['Mo', 'Mep'],
  'min': 0.0,
  'max': 80.0,
  'increment': 5.0},
 'hydrophobic': {'monomers': ['Ni',
   'Phe',
   'Do',
   'Bam',
   'Oct',
   'Olam',
   'Bmam',
   'Tmb'],
  'min': 10.0,
  'max': 45.0,
  'increment': 5.0}}

In [5]:
cols = ['mon_abb', 'SMILES', 'min_mon_wt', 'max_mon_wt', 'increments', 'molar_mass']
mon_df = df[cols].rename(columns={'min_mon_wt': 'min', 'max_mon_wt': 'max', 'increments': 'increment'})
monomer_properties = mon_df.set_index('mon_abb').T.to_dict('dict')
monomer_properties

{'Tma': {'SMILES': 'C[N+](C)(C)CCCNC(=O)C=C',
  'min': 10.0,
  'max': 90.0,
  'increment': 5.0,
  'molar_mass': 206.71},
 'Aeg': {'SMILES': '[Cl-].NC(=[NH2+])NCCNC(=O)C=C',
  'min': 0.0,
  'max': 20.0,
  'increment': 5.0,
  'molar_mass': 192.65},
 'Mo': {'SMILES': 'C=CC(=O)N1CCOCC1',
  'min': 0.0,
  'max': 90.0,
  'increment': 5.0,
  'molar_mass': 141.17},
 'Mep': {'SMILES': 'C=CC(=O)NCCCOC',
  'min': 0.0,
  'max': 90.0,
  'increment': 5.0,
  'molar_mass': 143.18},
 'Ni': {'SMILES': 'CC(C)NC(=O)C=C',
  'min': 0.0,
  'max': 40.0,
  'increment': 5.0,
  'molar_mass': 113.16},
 'Phe': {'SMILES': 'C=CC(=O)Nc1ccccc1',
  'min': 0.0,
  'max': 40.0,
  'increment': 5.0,
  'molar_mass': 147.17},
 'Do': {'SMILES': 'CCCCCCCCCCCCNC(=O)C=C',
  'min': 0.0,
  'max': 30.0,
  'increment': 5.0,
  'molar_mass': 239.4},
 'Bam': {'SMILES': 'CCCCOCNC(=O)C=C',
  'min': 0.0,
  'max': 40.0,
  'increment': 5.0,
  'molar_mass': 127.18},
 'Oct': {'SMILES': 'CCCCCCCCNC(=O)C=C',
  'min': 0.0,
  'max': 30.0,
  'increm

In [6]:
def get_range(key, dictionary):
    return range(int(dictionary[key]['min']), int(dictionary[key]['max']) + 1, int(dictionary[key]['increment']))

In [7]:
def product(*iterables, repeat=1):
    # product('ABCD', 'xy') → Ax Ay Bx By Cx Cy Dx Dy
    # product(range(2), repeat=3) → 000 001 010 011 100 101 110 111

    pools = [tuple(pool) for pool in iterables] * repeat

    result = [[]]
    for pool in pools:
        result = [x+[y] for x in result for y in pool]

    for prod in result:
        yield prod

In [8]:
def combinations(iterable, r):
    # combinations('ABCD', 2) → AB AC AD BC BD CD
    # combinations(range(4), 3) → 012 013 023 123

    pool = tuple(iterable)
    n = len(pool)
    if r > n:
        return
    indices = list(range(r))

    yield list(pool[i] for i in indices)
    while True:
        for i in reversed(range(r)):
            if indices[i] != i + n - r:
                break
        else:
            return
        indices[i] += 1
        for j in range(i+1, r):
            indices[j] = indices[j-1] + 1
        yield list(pool[i] for i in indices)

In [9]:
def generate_weights(monomer_ranges, target):
    # Generate all possible combinations using itertools.product
    possible_combinations = list(product(*monomer_ranges))
    
    # Filter combinations to ensure the sum equals 100%
    valid_combinations = [combo for combo in possible_combinations if sum(combo) == target]
    
    return valid_combinations

In [10]:
def generate_monomer_combinations(monomers, limit = None):
    
    all_combinations = []

    if limit is None:
        r_upper = len(monomers)
    else:
        r_upper = limit
    
    for r in range(1, r_upper + 1):
        all_combinations.extend(combinations(monomers, r))
    
    return all_combinations

In [11]:
monomer_class_ranges = [get_range('cationic', monomer_classes), get_range('hydrophilic', monomer_classes), get_range('hydrophobic', monomer_classes)]

In [12]:
monomer_class_distributions = generate_weights(monomer_class_ranges, 100)
monomer_class_distributions

[[10, 45, 45],
 [10, 50, 40],
 [10, 55, 35],
 [10, 60, 30],
 [10, 65, 25],
 [10, 70, 20],
 [10, 75, 15],
 [10, 80, 10],
 [15, 40, 45],
 [15, 45, 40],
 [15, 50, 35],
 [15, 55, 30],
 [15, 60, 25],
 [15, 65, 20],
 [15, 70, 15],
 [15, 75, 10],
 [20, 35, 45],
 [20, 40, 40],
 [20, 45, 35],
 [20, 50, 30],
 [20, 55, 25],
 [20, 60, 20],
 [20, 65, 15],
 [20, 70, 10],
 [25, 30, 45],
 [25, 35, 40],
 [25, 40, 35],
 [25, 45, 30],
 [25, 50, 25],
 [25, 55, 20],
 [25, 60, 15],
 [25, 65, 10],
 [30, 25, 45],
 [30, 30, 40],
 [30, 35, 35],
 [30, 40, 30],
 [30, 45, 25],
 [30, 50, 20],
 [30, 55, 15],
 [30, 60, 10],
 [35, 20, 45],
 [35, 25, 40],
 [35, 30, 35],
 [35, 35, 30],
 [35, 40, 25],
 [35, 45, 20],
 [35, 50, 15],
 [35, 55, 10],
 [40, 15, 45],
 [40, 20, 40],
 [40, 25, 35],
 [40, 30, 30],
 [40, 35, 25],
 [40, 40, 20],
 [40, 45, 15],
 [40, 50, 10],
 [45, 10, 45],
 [45, 15, 40],
 [45, 20, 35],
 [45, 25, 30],
 [45, 30, 25],
 [45, 35, 20],
 [45, 40, 15],
 [45, 45, 10],
 [50, 5, 45],
 [50, 10, 40],
 [50, 15, 3

In [13]:
monomer_combinations = {}
limits = {'hydrophobic': 3}

for key in monomer_classes.keys():
    if key in limits:
        monomer_combinations[key] = generate_monomer_combinations(monomer_classes[key]['monomers'], limits[key])
    else: 
        monomer_combinations[key] = generate_monomer_combinations(monomer_classes[key]['monomers'])
        
monomer_combinations

{'cationic': [['Tma'], ['Aeg'], ['Tma', 'Aeg']],
 'hydrophilic': [['Mo'], ['Mep'], ['Mo', 'Mep']],
 'hydrophobic': [['Ni'],
  ['Phe'],
  ['Do'],
  ['Bam'],
  ['Oct'],
  ['Olam'],
  ['Bmam'],
  ['Tmb'],
  ['Ni', 'Phe'],
  ['Ni', 'Do'],
  ['Ni', 'Bam'],
  ['Ni', 'Oct'],
  ['Ni', 'Olam'],
  ['Ni', 'Bmam'],
  ['Ni', 'Tmb'],
  ['Phe', 'Do'],
  ['Phe', 'Bam'],
  ['Phe', 'Oct'],
  ['Phe', 'Olam'],
  ['Phe', 'Bmam'],
  ['Phe', 'Tmb'],
  ['Do', 'Bam'],
  ['Do', 'Oct'],
  ['Do', 'Olam'],
  ['Do', 'Bmam'],
  ['Do', 'Tmb'],
  ['Bam', 'Oct'],
  ['Bam', 'Olam'],
  ['Bam', 'Bmam'],
  ['Bam', 'Tmb'],
  ['Oct', 'Olam'],
  ['Oct', 'Bmam'],
  ['Oct', 'Tmb'],
  ['Olam', 'Bmam'],
  ['Olam', 'Tmb'],
  ['Bmam', 'Tmb'],
  ['Ni', 'Phe', 'Do'],
  ['Ni', 'Phe', 'Bam'],
  ['Ni', 'Phe', 'Oct'],
  ['Ni', 'Phe', 'Olam'],
  ['Ni', 'Phe', 'Bmam'],
  ['Ni', 'Phe', 'Tmb'],
  ['Ni', 'Do', 'Bam'],
  ['Ni', 'Do', 'Oct'],
  ['Ni', 'Do', 'Olam'],
  ['Ni', 'Do', 'Bmam'],
  ['Ni', 'Do', 'Tmb'],
  ['Ni', 'Bam', 'Oct'],
  ['Ni',

In [14]:
polymer_combinations = list(product(*[monomer_combinations['cationic'], monomer_combinations['hydrophilic'], monomer_combinations['hydrophobic']]))
polymer_combinations

[[['Tma'], ['Mo'], ['Ni']],
 [['Tma'], ['Mo'], ['Phe']],
 [['Tma'], ['Mo'], ['Do']],
 [['Tma'], ['Mo'], ['Bam']],
 [['Tma'], ['Mo'], ['Oct']],
 [['Tma'], ['Mo'], ['Olam']],
 [['Tma'], ['Mo'], ['Bmam']],
 [['Tma'], ['Mo'], ['Tmb']],
 [['Tma'], ['Mo'], ['Ni', 'Phe']],
 [['Tma'], ['Mo'], ['Ni', 'Do']],
 [['Tma'], ['Mo'], ['Ni', 'Bam']],
 [['Tma'], ['Mo'], ['Ni', 'Oct']],
 [['Tma'], ['Mo'], ['Ni', 'Olam']],
 [['Tma'], ['Mo'], ['Ni', 'Bmam']],
 [['Tma'], ['Mo'], ['Ni', 'Tmb']],
 [['Tma'], ['Mo'], ['Phe', 'Do']],
 [['Tma'], ['Mo'], ['Phe', 'Bam']],
 [['Tma'], ['Mo'], ['Phe', 'Oct']],
 [['Tma'], ['Mo'], ['Phe', 'Olam']],
 [['Tma'], ['Mo'], ['Phe', 'Bmam']],
 [['Tma'], ['Mo'], ['Phe', 'Tmb']],
 [['Tma'], ['Mo'], ['Do', 'Bam']],
 [['Tma'], ['Mo'], ['Do', 'Oct']],
 [['Tma'], ['Mo'], ['Do', 'Olam']],
 [['Tma'], ['Mo'], ['Do', 'Bmam']],
 [['Tma'], ['Mo'], ['Do', 'Tmb']],
 [['Tma'], ['Mo'], ['Bam', 'Oct']],
 [['Tma'], ['Mo'], ['Bam', 'Olam']],
 [['Tma'], ['Mo'], ['Bam', 'Bmam']],
 [['Tma'], ['Mo'],

In [15]:
def monomer_class_ranges(monomer_class_in_poly, full_monomer_class):
    
    monomer_weight_ranges = []
    for mon in full_monomer_class:
        if mon in monomer_class_in_poly:
            monomer_weight_ranges.append(get_range(mon, monomer_properties))
        else:
            monomer_weight_ranges.append(range(0,1,5))

    return monomer_weight_ranges

In [16]:
def generate_polymer_weights(full_combo, polymer_combo, distribution):
    weights = []

    weights.append([[distribution]])
    
    for i in range(len(polymer_combo)):
        weights.append(generate_weights(monomer_class_ranges(polymer_combo[i], full_combo[i]), distribution[i]))
    return weights

In [17]:
def compute_mol(combo_list, flattened_full_combo, molar_masses):
    weights = np.array(combo_list[1:])

    weights = weights / molar_masses
    weights = weights / weights.sum()

    idx = weights.nonzero()    
    weights = weights[idx].tolist()

    combo_list.append(np.array(flattened_full_combo)[idx].tolist())
    combo_list.append([monomer_properties[mon]['SMILES'] for mon in np.array(flattened_full_combo)[idx]])
    combo_list.append(weights)

    return combo_list

In [18]:
# DEMO
# full_combo = [monomer_classes[key]['monomers'] for key in monomer_classes.keys()]
# flattened_full_combo = sum(full_combo,[])

# molar_masses = np.array([monomer_properties[mon]['molar_mass'] for mon in flattened_full_combo])


# polymer_combo = [['Tma', 'Aeg'], ['Mo'], ['Phe', 'Bam', 'Oct']]
# distribution = [15, 60, 25]
# generate_polymer_weights(full_combo, polymer_combo, distribution)

# [list(itertools.chain(*row)) for row in product(*generate_polymer_weights(full_combo,polymer_combo, distribution))]
# [compute_mol(list(itertools.chain(*row)), flattened_full_combo, molar_masses) for row in product(*generate_polymer_weights(full_combo,polymer_combo, distribution))]

In [19]:
full_combo = [monomer_classes[key]['monomers'] for key in monomer_classes.keys()]

flattened_full_combo = sum(full_combo,[])
molar_masses = np.array([monomer_properties[mon]['molar_mass'] for mon in flattened_full_combo])


col_names = full_combo.copy()
col_names.insert(0, ["class_distribution"])
col_names.append(["monomers"])
col_names.append(["SMILES"])
col_names.append(["mol_distribution"])

running_list = [sum(col_names,[])]

num = 1

for polymer_combo in polymer_combinations:
    print("Polymer Combination: " + str(num) + " / " + str(len(polymer_combinations)))
    num += 1
    for distribution in monomer_class_distributions:
        running_list.extend([compute_mol(list(itertools.chain(*row)), flattened_full_combo, molar_masses) 
                             for row in product(*generate_polymer_weights(full_combo, polymer_combo, distribution))])

Polymer Combination: 1 / 828
Polymer Combination: 2 / 828
Polymer Combination: 3 / 828
Polymer Combination: 4 / 828
Polymer Combination: 5 / 828
Polymer Combination: 6 / 828
Polymer Combination: 7 / 828
Polymer Combination: 8 / 828
Polymer Combination: 9 / 828
Polymer Combination: 10 / 828
Polymer Combination: 11 / 828
Polymer Combination: 12 / 828
Polymer Combination: 13 / 828
Polymer Combination: 14 / 828
Polymer Combination: 15 / 828
Polymer Combination: 16 / 828
Polymer Combination: 17 / 828
Polymer Combination: 18 / 828
Polymer Combination: 19 / 828
Polymer Combination: 20 / 828
Polymer Combination: 21 / 828
Polymer Combination: 22 / 828
Polymer Combination: 23 / 828
Polymer Combination: 24 / 828
Polymer Combination: 25 / 828
Polymer Combination: 26 / 828
Polymer Combination: 27 / 828
Polymer Combination: 28 / 828
Polymer Combination: 29 / 828
Polymer Combination: 30 / 828
Polymer Combination: 31 / 828
Polymer Combination: 32 / 828
Polymer Combination: 33 / 828
Polymer Combination

In [20]:
df = pd.DataFrame(data=running_list[1:],columns=running_list[0])
df = df.rename(columns={col: col + '_wt' for col in flattened_full_combo})
df.insert(0, 'ID', 'ID' + (df.index + 1).astype(str))

df

,ID,class_distribution,Tma_wt,Aeg_wt,Mo_wt,Mep_wt,Ni_wt,Phe_wt,Do_wt,Bam_wt,Oct_wt,Olam_wt,Bmam_wt,Tmb_wt,monomers,SMILES,mol_distribution
0,ID1,"[10, 50, 40]",10,0,50,0,40,0,0,0,0,0,0,0,"[Tma, Mo, Ni]","[C[N+](C)(C)CCCNC(=O)C=C, C=CC(=O)N1CCOCC1, CC...","[0.06398715373187551, 0.4684700909511932, 0.46..."
1,ID2,"[10, 55, 35]",10,0,55,0,35,0,0,0,0,0,0,0,"[Tma, Mo, Ni]","[C[N+](C)(C)CCCNC(=O)C=C, C=CC(=O)N1CCOCC1, CC...","[0.06473784309939555, 0.52136273648026, 0.4138..."
2,ID3,"[10, 60, 30]",10,0,60,0,30,0,0,0,0,0,0,0,"[Tma, Mo, Ni]","[C[N+](C)(C)CCCNC(=O)C=C, C=CC(=O)N1CCOCC1, CC...","[0.06550635555466004, 0.5755111747554202, 0.35..."
3,ID4,"[10, 65, 25]",10,0,65,0,25,0,0,0,0,0,0,0,"[Tma, Mo, Ni]","[C[N+](C)(C)CCCNC(=O)C=C, C=CC(=O)N1CCOCC1, CC...","[0.06629333346479015, 0.6309606661705321, 0.30..."
4,ID5,"[10, 70, 20]",10,0,70,0,20,0,0,0,0,0,0,0,"[Tma, Mo, Ni]","[C[N+](C)(C)CCCNC(=O)C=C, C=CC(=O)N1CCOCC1, CC...","[0.06709945044125784, 0.6877586725578159, 0.24..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7026132,ID7026133,"[90, 0, 10]",90,0,0,0,0,0,0,0,0,0,5,5,"[Tma, Bmam, Tmb]","[C[N+](C)(C)CCCNC(=O)C=C, C=CC(=O)NCCCC, CC(C)...","[0.8805124490527483, 0.0643197451193003, 0.055..."
7026133,ID7026134,"[90, 0, 10]",90,0,0,0,0,0,0,0,0,0,10,0,"[Tma, Bmam]","[C[N+](C)(C)CCCNC(=O)C=C, C=CC(=O)NCCCC]","[0.8725271336951159, 0.12747286630488405]"
7026134,ID7026135,"[90, 0, 10]",90,0,0,0,0,0,0,0,0,5,0,5,"[Tma, Olam, Tmb]","[C[N+](C)(C)CCCNC(=O)C=C, CCCCCCCC/C=C\CCCCCCC...","[0.9104183369193181, 0.032540127680013874, 0.0..."
7026135,ID7026136,"[90, 0, 10]",90,0,0,0,0,0,0,0,0,5,5,0,"[Tma, Olam, Bmam]","[C[N+](C)(C)CCCNC(=O)C=C, CCCCCCCC/C=C\CCCCCCC...","[0.9018840092521299, 0.03223509415784967, 0.06..."


In [21]:
df.to_csv("polymer_combinations.csv", index=None, sep=',', mode='w')

In [22]:
# SANITY CHECK
# df['mol_distribution'].map(lambda x: sum(x)).value_counts()
# (df['monomers'].map(lambda x: len(x)) - df['SMILES'].map(lambda x: len(x))).value_counts()